<a href="https://colab.research.google.com/github/SaharMian/recsys-letterboxd/blob/main/Dataandeval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie recommendations on Letterboxd data, part 1: data, evaluation, popularity baseline and SVD

**Question:** How much better than a popularity baseline can matrix factorization and neural models do on Letterboxd ratings, and where do they fail?

**This notebook covers:**
1. A first look at the data
2. Sampling 3,000 users
3. Splitting each user's ratings into train / validation / test
4. The evaluation method (NDCG@10, Recall@10, coverage)
5. The popularity baseline
6. Matrix factorization (SVD-style) in PyTorch

The next notebooks cover Neural Collaborative Filtering and a two-tower model. Run the notebooks in order.

**Data:** [Letterboxd Film Ratings](https://www.kaggle.com/datasets/freeth/letterboxd-film-ratings) by freeth on Kaggle. Put `ratings.csv` and `films.csv` in a folder called `recsys` in Google Drive.

## Setup

Colab forgets all files when a session ends, so the data lives in Google Drive. This cell gives the notebook access to it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Check that the two data files are in the `recsys` folder.

In [2]:
import os
print(os.listdir('/content/drive/MyDrive/recsys'))

['films.csv', 'ratings.csv', 'sample_3000.parquet', 'split_3000.parquet', 'svd_model.pt', 'ncf_model.pt', 'title.basics.tsv', 'film_text_emb.npy', 'two_tower_model.pt', 'results_val.csv', 'app_data']


## A first look at the data

Reading only the first 1,000 rows is a quick way to see the structure before loading all 18 million. Each row is one rating: a user, a film and a star rating from 0.5 to 5.0.

Note that the file is sorted by user, so these first rows all come from one person. They tell us about the format, not about the data as a whole.

In [3]:
import pandas as pd

BASE = '/content/drive/MyDrive/recsys/'

ratings = pd.read_csv(BASE + 'ratings.csv', nrows=1000)
print(ratings.shape)
print(ratings.columns.tolist())
ratings.head()

(1000, 3)
['user_name', 'film_id', 'rating']


,user_name,film_id,rating
0,kurstboy,spider-man-2,4.5
1,kurstboy,tetsuo-the-iron-man,4.0
2,kurstboy,bram-stokers-dracula,4.5
3,kurstboy,poison-2023,4.5
4,kurstboy,the-rat-catcher-2023,3.5


`films.csv` has the film id, name, year and a poster link. It has **no genre or description**, which matters later for the two-tower model. `film_id` links the two files together.

A few rows in the file are malformed, so we skip them with `on_bad_lines='skip'`.

In [4]:
BASE = '/content/drive/MyDrive/recsys/'

films = pd.read_csv(BASE + 'films.csv', engine='python', on_bad_lines='skip')
print(films.shape)
print(films.columns.tolist())
films.head()

(354834, 4)
['film_id', 'film_name', 'year', 'poster_url']


,film_id,film_name,year,poster_url
0,anthem-of-a-teenage-prophet,Anthem of a Teenage Prophet,2018.0,https://a.ltrbxd.com/resized/film-poster/3/8/6...
1,elf-buddys-musical-christmas,Elf: Buddy's Musical Christmas,2014.0,https://a.ltrbxd.com/resized/film-poster/2/3/8...
2,look-at-us-now-mother,"Look at Us Now, Mother!",2016.0,https://a.ltrbxd.com/resized/film-poster/3/2/6...
3,fate-grand-order-absolute-demonic-front-babylonia,Fate/Grand Order Absolute Demonic Front: Babyl...,2019.0,https://a.ltrbxd.com/resized/film-poster/6/7/1...
4,memory-2022,Memory,2022.0,https://a.ltrbxd.com/resized/film-poster/7/3/4...


How many rows did we skip? We compare the line count of the file with the number of rows pandas read.

In [5]:
with open(BASE + 'films.csv', encoding='utf-8') as f:
    n_lines = sum(1 for _ in f) - 1

print("Lines in file:  ", n_lines)
print("Rows in films:  ", len(films))
print("Difference:     ", n_lines - len(films))

Lines in file:   355141
Rows in films:   354834
Difference:      307


307 of 355,141 rows (0.09%) could not be read. That is negligible, but worth noting.

## Load all ratings

The file is 595 MB with 18.2 million ratings. `user_name` and `film_id` are text strings repeated millions of times, so we store them as `category` (each unique value is stored once and rows point to it). Ratings only need `float32` instead of `float64`. Together this cuts memory use to about 215 MB, which fits in Colab.

In [6]:
print(f"File size: {os.path.getsize(BASE + 'ratings.csv') / 1e6:.0f} MB")

ratings = pd.read_csv(
    BASE + 'ratings.csv',
    dtype={'user_name': 'category', 'film_id': 'category', 'rating': 'float32'}
)
print(ratings.shape)
print(f"Memory: {ratings.memory_usage(deep=True).sum() / 1e6:.0f} MB")

File size: 595 MB
(18175545, 3)
Memory: 215 MB


## Basic statistics

How many ratings, users and films are there? *Sparsity* is the share of all possible user-film pairs that have no rating. Recommender systems exist to predict those missing ratings.

The count per rating value shows how people use the star scale: 3.0 to 4.0 are the most common, so most ratings are "okay to good".

In [7]:
n_ratings = len(ratings)
n_users = ratings['user_name'].nunique()
n_films = ratings['film_id'].nunique()

print(f"Ratings:  {n_ratings:,}")
print(f"Users:    {n_users:,}")
print(f"Films:    {n_films:,}")
print(f"Sparsity: {1 - n_ratings / (n_users * n_films):.2%} of all user-film pairs have no rating")

print(ratings['rating'].value_counts().sort_index())

Ratings:  18,175,545
Users:    11,061
Films:    355,140
Sparsity: 99.54% of all user-film pairs have no rating
rating
0.5     309105
1.0     615883
1.5     636460
2.0    1523578
2.5    1753268
3.0    3492302
3.5    3310119
4.0    3631351
4.5    1519156
5.0    1384323
Name: count, dtype: int64


## Ratings per user and per film

For films the mean (51) is far above the median (3): a few films have thousands of ratings, while most have almost none. This **long tail** makes a popularity baseline strong, and makes films with few ratings hard for any model (the *cold start* problem). `observed=True` is needed because the columns are categories.

In [8]:
per_user = ratings.groupby('user_name', observed=True).size()
per_film = ratings.groupby('film_id', observed=True).size()

print("Ratings per user:")
print(per_user.describe())

print("\nRatings per film:")
print(per_film.describe())

print(f"\nFilms with only 1 rating: {(per_film == 1).mean():.1%}")
print(f"The 1,000 most rated films account for {per_film.nlargest(1000).sum() / len(ratings):.1%} of all ratings")

Ratings per user:
count    11061.000000
mean      1643.209927
std       1987.538284
min          1.000000
25%        562.000000
50%       1129.000000
75%       2088.000000
max      90755.000000
dtype: float64

Ratings per film:
count    355140.000000
mean         51.178513
std         307.481606
min           1.000000
25%           1.000000
50%           3.000000
75%          10.000000
max        8647.000000
dtype: float64

Films with only 1 rating: 31.6%
The 1,000 most rated films account for 24.8% of all ratings


## Choosing filters

Before sampling we check how much data different minimum thresholds would remove. What matters is the share of **ratings** kept, not the number of films: requiring at least 10 ratings per film keeps only 26% of the films but 96.1% of the ratings. For users, a minimum of 50 ratings costs nothing.

The most active user has 90,755 ratings, about 55 times the average. That is why we later cap the number of ratings per user.

In [9]:
print("FILMS: at least N ratings")
for n in [3, 5, 10, 20, 50, 75]:
    kept = per_film[per_film >= n]
    print(f"  N={n:>3}: {len(kept):>7,} films ({len(kept)/len(per_film):.0%}), "
          f"{kept.sum()/len(ratings):.1%} of all ratings kept")

print("\nUSERS: at least N ratings")
for n in [20, 50, 100, 300, 562, 700, 1000]:
    kept = per_user[per_user >= n]
    print(f"  N={n:>3}: {len(kept):>6,} users ({len(kept)/len(per_user):.0%}), "
          f"{kept.sum()/len(ratings):.1%} of all ratings kept")

print("\nThe 10 most active users:")
print(per_user.nlargest(10))
print(f"They account for {per_user.nlargest(10).sum()/len(ratings):.1%} of all ratings")

FILMS: at least N ratings
  N=  3: 191,716 films (54%), 98.8% of all ratings kept
  N=  5: 142,374 films (40%), 97.9% of all ratings kept
  N= 10:  94,078 films (26%), 96.1% of all ratings kept
  N= 20:  61,091 films (17%), 93.7% of all ratings kept
  N= 50:  34,185 films (10%), 89.1% of all ratings kept
  N= 75:  26,503 films (7%), 86.6% of all ratings kept

USERS: at least N ratings
  N= 20: 10,983 users (99%), 100.0% of all ratings kept
  N= 50: 10,882 users (98%), 100.0% of all ratings kept
  N=100: 10,719 users (97%), 99.9% of all ratings kept
  N=300:  9,750 users (88%), 98.8% of all ratings kept
  N=562:  8,299 users (75%), 95.4% of all ratings kept
  N=700:  7,545 users (68%), 92.7% of all ratings kept
  N=1000:  6,099 users (55%), 86.0% of all ratings kept

The 10 most active users:
user_name
acervofilmico      90755
punq               35541
ghettominibar      27236
loureviews         25021
thecinema4pylon    21652
allisoncm          21246
sbsiceland         21151
voidness    

## Sampling 3,000 users

Training on all 18 million ratings would be slow and unnecessary, so we sample users. The order of the steps matters:
1. Pick 3,000 random users who have at least 50 ratings.
2. Cap each user at 5,000 randomly chosen ratings, so no single person dominates the training.
3. Keep only films with at least 5 ratings **in the sample**. The film filter has to come last, otherwise films would end up with too few ratings to train and evaluate on.

The seed (42) makes the sample reproducible.

In [10]:
import numpy as np

MIN_USER_RATINGS = 50
MAX_USER_RATINGS = 5000
N_USERS = 3000
MIN_FILM_RATINGS = 5
SEED = 42

rng = np.random.default_rng(SEED)

# 1. Randomly pick N_USERS among users with enough ratings
eligible = per_user[per_user >= MIN_USER_RATINGS].index.to_numpy()
chosen = rng.choice(eligible, size=N_USERS, replace=False)
sample = ratings[ratings['user_name'].isin(chosen)]

# 2. Cap per user: shuffle the rows, keep the first MAX_USER_RATINGS per user
sample = sample.sample(frac=1, random_state=SEED)
sample = sample[sample.groupby('user_name', observed=True).cumcount() < MAX_USER_RATINGS]

# 3. Filter films, counted in the sample
film_counts = sample['film_id'].value_counts()
keep_films = film_counts[film_counts >= MIN_FILM_RATINGS].index
sample = sample[sample['film_id'].isin(keep_films)].copy()

# 4. Drop unused categories (otherwise the columns still carry all 355,000 films)
for col in ['user_name', 'film_id']:
    sample[col] = sample[col].cat.remove_unused_categories()

print(f"Users:   {sample['user_name'].nunique():,}")
print(f"Films:   {sample['film_id'].nunique():,}")
print(f"Ratings: {len(sample):,}  ({len(sample)/len(ratings):.1%} of the original)")
print(f"Ratings per user (median): {sample.groupby('user_name', observed=True).size().median():.0f}")

Users:   3,000
Films:   61,555
Ratings: 4,439,328  (24.4% of the original)
Ratings per user (median): 1114


## Is the sample representative?

We compare the rating distribution of the sample with the original. All differences are below 0.6 percentage points. The sample has slightly more high ratings (4.5 and 5.0), probably because the film filter removes obscure films, which tend to get lower ratings.

The sample is saved to Drive as a parquet file, which is smaller than CSV and keeps the column types.

In [11]:
orig = ratings['rating'].value_counts(normalize=True).sort_index()
samp = sample['rating'].value_counts(normalize=True).sort_index()
print(pd.DataFrame({'original': orig, 'sample': samp}).round(3))

sample.to_parquet(BASE + 'sample_3000.parquet', index=False)
print("Saved!")

        original  sample
rating                  
0.5        0.017   0.016
1.0        0.034   0.032
1.5        0.035   0.034
2.0        0.084   0.080
2.5        0.096   0.092
3.0        0.192   0.187
3.5        0.182   0.183
4.0        0.200   0.205
4.5        0.084   0.089
5.0        0.076   0.082
Saved!


## Train / validation / test split

Each user's ratings are shuffled and split 80/10/10, so every user appears in all three parts:
- **Train:** the models learn from these ratings.
- **Validation:** used to choose settings, for example which epoch to keep.
- **Test:** touched only once, at the very end.

We shuffle first because the file is sorted by user, so an unshuffled split would not be random. "Liked" means a rating of 4.0 or higher.

Checks: all 3,000 users have at least one liked film in test, and 0% of the test ratings are on films missing from train. That means film cold start is **not** visible in this test set, so we test it separately in the two-tower notebook.

Limitation: the data has no timestamps, so the split is random per user and not chronological.

In [12]:
# Shuffle all rows, then split each user's ratings 80/10/10
data = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)

grp = data.groupby('user_name', observed=True)
frac = grp.cumcount() / grp['rating'].transform('size')

data['split'] = 'train'
data.loc[frac >= 0.8, 'split'] = 'val'
data.loc[frac >= 0.9, 'split'] = 'test'

print(data['split'].value_counts())

test_liked = data[(data['split'] == 'test') & (data['rating'] >= 4.0)]
print(f"\nUsers with at least 1 liked film in test: {test_liked['user_name'].nunique():,} of {data['user_name'].nunique():,}")

train_films = set(data.loc[data['split'] == 'train', 'film_id'].unique())
test_films = data.loc[data['split'] == 'test', 'film_id']
print(f"Test ratings on films missing from train: {(~test_films.isin(train_films)).mean():.2%}")

data.to_parquet(BASE + 'split_3000.parquet', index=False)

split
train    3552662
val       444086
test      442580
Name: count, dtype: int64

Users with at least 1 liked film in test: 3,000 of 3,000
Test ratings on films missing from train: 0.00%


## Evaluation

All models are measured in the same way. For each user, a model ranks all films the user has not already rated in train, and we look at its top 10. We then check how many of the films the user **liked** (rating of 4.0 or higher) in the held-out part appear in that list.

- **NDCG@10:** ranking quality. Hits near the top count more. 1.0 is perfect.
- **Recall@10:** the share of the user's liked held-out films that appear in the top 10. Many users have 50 or more liked films, so this can never be close to 1.
- **Coverage:** the share of all films that are ever recommended to anyone. A low value means the model recommends the same few films to everybody.

In [13]:
K = 10        # we recommend 10 films per user
LIKE = 4.0    # a rating of 4.0 or higher counts as "liked"

# Integer ids for users and films (the models need numbers, not text)
data['u'] = data['user_name'].cat.codes
data['i'] = data['film_id'].cat.codes
n_users, n_films = data['u'].nunique(), data['i'].nunique()

train = data[data['split'] == 'train']
val   = data[data['split'] == 'val']
test  = data[data['split'] == 'test']

# Films each user has already rated in train (these are never recommended)
seen_train = train.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def relevant_sets(df):
    liked = df[df['rating'] >= LIKE]
    return liked.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def evaluate(recommend_fn, eval_df, seen):
    relevant = relevant_sets(eval_df)
    ndcgs, recalls, recommended = [], [], set()
    for user, rel in relevant.items():
        recs = recommend_fn(user, seen.get(user, set()), K)
        recommended.update(recs)
        hits = [1 if f in rel else 0 for f in recs]
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcgs.append(dcg / idcg)
        recalls.append(sum(hits) / len(rel))
    return {
        'NDCG@10': np.mean(ndcgs),
        'Recall@10': np.mean(recalls),
        'Coverage': len(recommended) / data['film_id'].nunique(),
    }

## Popularity baseline

The simplest possible model: rank films by how many users liked them in train, and recommend the most popular ones the user has not seen. It is not personal, but it is hard to beat in this data because a few films get a large share of all ratings. Every other model has to beat it.

In [14]:
# Films ranked by number of likes in train
pop_rank = train[train['rating'] >= LIKE]['film_id'].value_counts().index.tolist()

def recommend_pop(user, seen, k):
    out = []
    for f in pop_rank:
        if f not in seen:
            out.append(f)
            if len(out) == k:
                break
    return out

print(evaluate(recommend_pop, val, seen_train))

{'NDCG@10': np.float64(0.18405938695253699), 'Recall@10': np.float64(0.03855764705308532), 'Coverage': 0.001364633254812769}


## Two ways to count popularity

We can rank films by the number of likes (rating of 4.0 or higher) or by the number of ratings of any kind. Counting likes is slightly better (NDCG 0.1841 vs 0.1745), so that is our baseline.

In [15]:
def make_pop_fn(rank):
    def fn(user, seen, k):
        out = []
        for f in rank:
            if f not in seen:
                out.append(f)
                if len(out) == k:
                    break
        return out
    return fn

pop_all = train['film_id'].value_counts().index.tolist()

results = {}
results['Popularity (likes)']       = evaluate(make_pop_fn(pop_rank), val, seen_train)
results['Popularity (all ratings)'] = evaluate(make_pop_fn(pop_all),  val, seen_train)

pd.DataFrame(results).T.round(4)

,NDCG@10,Recall@10,Coverage
Popularity (likes),0.1841,0.0386,0.0014
Popularity (all ratings),0.1745,0.0377,0.0013


## SVD: matrix factorization

Every user and every film gets a short vector (here 32 numbers). The predicted rating is

`global mean + user bias + film bias + (user vector · film vector)`

The user vector describes taste, the film vector describes the film's properties, and the dot product is high when they match. The biases capture that some users rate generously and some films are liked by everyone, so the vectors can focus on what is personal.

Training adjusts the vectors so that the predictions are close to the real ratings in train. Regularization penalizes large vectors so the model does not memorize the training data. This is the model described by Koren, Bell and Volinsky (2009).

Important: this model is trained to predict **ratings**, not to rank films. It turns out that matters a lot. Use a GPU runtime (T4) for this part.

In [16]:
import torch
import torch.nn as nn

device = 'cuda'
print("GPU available:", torch.cuda.is_available())

GPU available: True


## Model and training

Each epoch goes through all train ratings in random batches. For each batch the model predicts ratings, we measure the error (squared error plus the regularization penalty) and adjust the vectors with the Adam optimizer. After every epoch we measure **RMSE** on the validation set, which is the typical error in stars.

For reference, always guessing the global mean gives an RMSE of 1.042. The trained model reaches about 0.709 (32% better) and then levels off, with no sign of overfitting.

In [17]:
def to_t(df):
    return (torch.tensor(df['u'].values, dtype=torch.long, device=device),
            torch.tensor(df['i'].values, dtype=torch.long, device=device),
            torch.tensor(df['rating'].values, dtype=torch.float32, device=device))

tr_u, tr_i, tr_r = to_t(data[data['split'] == 'train'])
va_u, va_i, va_r = to_t(data[data['split'] == 'val'])

class MF(nn.Module):
    """Predicted rating = mean + user bias + film bias + user vector . film vector"""
    def __init__(self, n_users, n_films, d, mu):
        super().__init__()
        self.P, self.Q = nn.Embedding(n_users, d), nn.Embedding(n_films, d)
        self.bu, self.bi = nn.Embedding(n_users, 1), nn.Embedding(n_films, 1)
        self.mu = mu
        nn.init.normal_(self.P.weight, std=0.1)
        nn.init.normal_(self.Q.weight, std=0.1)
        nn.init.zeros_(self.bu.weight)
        nn.init.zeros_(self.bi.weight)
    def forward(self, u, i):
        return (self.mu + self.bu(u).squeeze(1) + self.bi(i).squeeze(1)
                + (self.P(u) * self.Q(i)).sum(1))

# vector size, learning rate, regularization strength, batch size, epochs
D, LR, LAM, BATCH, EPOCHS = 32, 0.005, 0.05, 16384, 15

torch.manual_seed(SEED)
model = MF(n_users, n_films, D, tr_r.mean().item()).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)

def rmse(u, i, r):
    model.eval()
    with torch.no_grad():
        return ((model(u, i) - r) ** 2).mean().sqrt().item()

print(f"RMSE if we always guess the mean: {((va_r - tr_r.mean()) ** 2).mean().sqrt().item():.3f}")

n = len(tr_r)
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(n, device=device)
    for s in range(0, n, BATCH):
        idx = perm[s:s + BATCH]
        u, i, r = tr_u[idx], tr_i[idx], tr_r[idx]
        reg = LAM * (model.P(u).pow(2).sum(1) + model.Q(i).pow(2).sum(1)
                     + model.bu(u).pow(2).squeeze(1) + model.bi(i).pow(2).squeeze(1)).mean()
        loss = ((model(u, i) - r) ** 2).mean() + reg
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(f"Epoch {epoch + 1:>2}: val RMSE {rmse(va_u, va_i, va_r):.4f}")

RMSE if we always guess the mean: 1.042
Epoch  1: val RMSE 0.7637
Epoch  2: val RMSE 0.7286
Epoch  3: val RMSE 0.7186
Epoch  4: val RMSE 0.7138
Epoch  5: val RMSE 0.7114
Epoch  6: val RMSE 0.7101
Epoch  7: val RMSE 0.7095
Epoch  8: val RMSE 0.7093
Epoch  9: val RMSE 0.7090
Epoch 10: val RMSE 0.7091
Epoch 11: val RMSE 0.7088
Epoch 12: val RMSE 0.7087
Epoch 13: val RMSE 0.7089
Epoch 14: val RMSE 0.7088
Epoch 15: val RMSE 0.7087


## Ranking all films and evaluating

RMSE says how well the model guesses ratings, but what we care about is ranking. For each user we compute a predicted rating for all 61,555 films, remove the films already rated in train, and take the 10 highest. This is done in chunks of 500 users to fit in GPU memory.

The parameter `min_count` can exclude films with few ratings in train from the candidates. Here it is 0 (no filter); we come back to it below.

In [18]:
# Number of train ratings per film (used to filter out rarely rated films)
film_train_count = torch.tensor(np.bincount(train['i'].values, minlength=n_films), device=device)

def recommend_all(model, exclude=('train',), k=K, chunk=500, min_count=0):
    ex = data[data['split'].isin(exclude)]
    ex_u = torch.tensor(ex['u'].values, dtype=torch.long, device=device)
    ex_i = torch.tensor(ex['i'].values, dtype=torch.long, device=device)
    users = data['user_name'].cat.categories
    films = data['film_id'].cat.categories
    allowed = film_train_count >= min_count
    model.eval()
    recs = {}
    with torch.no_grad():
        for start in range(0, n_users, chunk):
            end = min(start + chunk, n_users)
            u = torch.arange(start, end, device=device)
            scores = model.mu + model.bu(u) + model.bi.weight.T + model.P(u) @ model.Q.weight.T
            scores[:, ~allowed] = float('-inf')                      # too few ratings in train
            sel = (ex_u >= start) & (ex_u < end)
            scores[ex_u[sel] - start, ex_i[sel]] = float('-inf')     # already seen
            top = scores.topk(k, dim=1).indices.cpu().numpy()
            for row, uid in enumerate(range(start, end)):
                recs[users[uid]] = [films[j] for j in top[row]]
    return recs

recs_svd = recommend_all(model)

results = {}
results['Popularity (likes)'] = evaluate(recommend_pop, val, seen_train)
results['SVD (PyTorch)'] = evaluate(lambda user, seen, k: recs_svd[user], val, seen_train)

pd.DataFrame(results).T.round(4)

,NDCG@10,Recall@10,Coverage
Popularity (likes),0.1841,0.0386,0.0014
SVD (PyTorch),0.0086,0.0022,0.0323


## Diagnosis: which films does SVD recommend?

SVD scores far below the popularity baseline (NDCG@10 0.0086 vs 0.184), which is suspicious. To see why, we count how many train ratings the recommended films have, compared with the popularity recommendations and with the films users actually liked in validation.

Result: SVD recommends films with a **median of 12 train ratings**, while the films users really liked have a median of 463 and the popularity list has 1630. The model was trained to predict ratings, so it is rewarded for being right about films with few, enthusiastic ratings (for example concert films from K-pop groups such as BTS and Blackpink). Those films get a very high predicted score from a handful of fans, but almost nobody else has rated them. This is a form of selection bias.

In [19]:
tc = train['film_id'].value_counts().to_dict()   # number of train ratings per film

def median_count(recs):
    """Median number of train ratings among all recommended films"""
    counts = [tc.get(f, 0) for films in recs.values() for f in films]
    return int(np.median(counts))

users = data['user_name'].cat.categories
recs_pop = {u: recommend_pop(u, seen_train.get(u, set()), K) for u in users}

liked_val = val[val['rating'] >= LIKE]['film_id']
median_liked = int(liked_val.map(lambda f: tc.get(f, 0)).median())

print(f"Median train ratings, SVD recommendations:        {median_count(recs_svd)}")
print(f"Median train ratings, films liked in validation:  {median_liked}")
print(f"Median train ratings, popularity recommendations: {median_count(recs_pop)}")

Median train ratings, SVD recommendations:        12
Median train ratings, films liked in validation:  463
Median train ratings, popularity recommendations: 1630


## Fix: require a minimum number of ratings (`min_count`)

A simple fix is to only allow films with at least `min_count` train ratings as candidates. This is a common practice in recommender systems: a film with 5 ratings is too uncertain to recommend. We try different thresholds and look at the validation metrics.

NDCG@10 rises from 0.0086 (no filter) to 0.1374 at 500 ratings, so most of the gap to popularity was caused by the obscure films. Coverage falls as the threshold rises, since fewer films are allowed.

In [20]:
for n in [0, 10, 50, 100, 200, 500]:
    recs = recommend_all(model, min_count=n)
    res = evaluate(lambda user, seen, k: recs[user], val, seen_train)
    print(f"min {n:>3} ratings: NDCG@10 {res['NDCG@10']:.4f}, "
          f"Recall@10 {res['Recall@10']:.4f}, Coverage {res['Coverage']:.4f}")

min   0 ratings: NDCG@10 0.0086, Recall@10 0.0022, Coverage 0.0323
min  10 ratings: NDCG@10 0.0149, Recall@10 0.0033, Coverage 0.0265
min  50 ratings: NDCG@10 0.0479, Recall@10 0.0106, Coverage 0.0147
min 100 ratings: NDCG@10 0.0685, Recall@10 0.0148, Coverage 0.0112
min 200 ratings: NDCG@10 0.0955, Recall@10 0.0200, Coverage 0.0095
min 500 ratings: NDCG@10 0.1374, Recall@10 0.0279, Coverage 0.0065


### Higher thresholds

We continue upwards to find where it stops helping. The number of films left shows what the filter costs. At 2000 ratings no films remain at all in this 3,000-user sample, so that value is invalid and only shown as a limit.

In [21]:
for n in [500, 750, 1000, 1500, 2000]:
    left = int((film_train_count >= n).sum())
    if left < K:
        print(f"min {n:>4} ratings: only {left} films left, not enough to recommend {K}")
        continue
    recs = recommend_all(model, min_count=n)
    res = evaluate(lambda user, seen, k: recs[user], val, seen_train)
    print(f"min {n:>4} ratings ({left:>5} films left): NDCG@10 {res['NDCG@10']:.4f}, "
          f"Recall@10 {res['Recall@10']:.4f}, Coverage {res['Coverage']:.4f}")

min  500 ratings ( 1513 films left): NDCG@10 0.1374, Recall@10 0.0279, Coverage 0.0065
min  750 ratings (  738 films left): NDCG@10 0.1628, Recall@10 0.0335, Coverage 0.0051
min 1000 ratings (  400 films left): NDCG@10 0.1757, Recall@10 0.0365, Coverage 0.0038
min 1500 ratings (   52 films left): NDCG@10 0.1929, Recall@10 0.0406, Coverage 0.0010
min 2000 ratings: only 0 films left, not enough to recommend 10


## Save validation results and the model

We save the validation results to `results_val.csv`, so the other notebooks can add their rows to the same table later. We keep the plain popularity baseline and SVD with three thresholds (no filter, 1000 and 1500 ratings). The model is also saved, together with its global mean `mu`, which is not part of `state_dict()`.

This notebook runs first, so it creates the file from scratch. The next notebooks (`nfc`, `twotower`) add rows to it.

In [22]:
rows = {}
rows['Popularity (likes)'] = evaluate(recommend_pop, val, seen_train)
for n in [0, 1000, 1500]:
    recs = recommend_all(model, min_count=n)
    rows[f'SVD (min {n} ratings)'] = evaluate(lambda user, seen, k: recs[user], val, seen_train)

df_val = pd.DataFrame(rows).T[['NDCG@10', 'Recall@10', 'Coverage']]
df_val.to_csv(BASE + 'results_val.csv')

torch.save({'state_dict': model.state_dict(), 'mu': model.mu}, BASE + 'svd_model.pt')
df_val.round(4)

,NDCG@10,Recall@10,Coverage
Popularity (likes),0.1841,0.0386,0.0014
SVD (min 0 ratings),0.0086,0.0022,0.0323
SVD (min 1000 ratings),0.1757,0.0365,0.0038
SVD (min 1500 ratings),0.1929,0.0406,0.0010


## Final evaluation on the test set

Until now we only used the validation set to make choices (model settings, the `min_count` threshold). The test set has not been touched. We now evaluate **once**, with the settings chosen on validation, so the numbers are an honest estimate of how the models do on new data.

One difference: at test time both train and validation ratings count as "already seen" for each user, so those films are excluded from the recommendations. The test results are saved to a separate file, `results_test.csv`.

Note: the test numbers are higher than the validation numbers for every model. This is because the validation protocol leaves the test ratings among the candidates, while at test time both train and validation ratings are excluded. Compare models within the same table, not across tables.

Main finding: SVD with the best threshold from validation (1500 ratings) is slightly better than popularity on validation (0.1929 vs 0.1841) but slightly worse on test (0.2377 vs 0.2562). The small advantage did not hold up, which is why the test set is kept separate.

In [23]:
import os

TEST_PATH = BASE + 'results_test.csv'
COLS = ['NDCG@10', 'Recall@10', 'Coverage']

def add_test_row(name, res):
    """Add (or replace) a row in results_test.csv"""
    if os.path.exists(TEST_PATH):
        df = pd.read_csv(TEST_PATH, index_col=0)
    else:
        df = pd.DataFrame(columns=COLS)
    df.loc[name] = [res[c] for c in COLS]
    df.to_csv(TEST_PATH)
    return df

# Films each user has seen in train + validation (excluded at test time)
seen_tv = data[data['split'].isin(['train', 'val'])] \
    .groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

In [24]:
# Popularity baseline
res = evaluate(recommend_pop, test, seen_tv)
add_test_row('Popularity (likes)', res)

# SVD with the thresholds chosen on validation
for n in [0, 1500]:
    recs = recommend_all(model, exclude=('train', 'val'), min_count=n)
    res = evaluate(lambda user, seen, k: recs[user], test, seen_tv)
    df_test = add_test_row(f'SVD (min {n} ratings)', res)

df_test.round(4)

,NDCG@10,Recall@10,Coverage
Popularity (likes),0.2562,0.0493,0.0024
SVD (min 0 ratings),0.0106,0.0022,0.0324
SVD (min 1500 ratings),0.2377,0.0459,0.0010
